# Dizajn skladišta podataka

Model se temelji na rezultatima bilježnica:

- `01_analiza_podataka.ipynb`
- `02_ciscenje_podataka.ipynb`

Dizajn slijedi Kimballov redoslijed:

1. odabir poslovnog procesa;
2. precizno određivanje granulacije;
3. definiranje dimenzija;
4. definiranje činjenica i mjera.

Skladište se oblikuje kao skup povezanih zvjezdastih modela. Razlog je to što stavka narudžbe, cijela narudžba, pojedinačno plaćanje, recenzija i mjesečni statistički sažetak nemaju jednaku granulaciju.

Vremenska komponenta modelira se pomoću:

- `dim_date`, čija je granulacija jedan kalendarski datum;
- `dim_time`, čija je granulacija jedna minuta unutar dana.

Razdvajanje datuma i vremena sprječava stvaranje velike datetime dimenzije, a istodobno omogućuje analize po satu, minuti, dijelu dana i drugim vremenskim obilježjima.


# Poslovna pitanja

## Prodaja

- Kako se broj prodanih stavki i ukupna vrijednost prodaje mijenjaju kroz vrijeme?
- Koji proizvodi, kategorije i prodavači ostvaruju najveću vrijednost prodaje?
- Koliki je trošak dostave prema proizvodu, kategoriji, prodavaču i lokaciji kupca?
- Koliko cijena pojedine stavke odstupa od cijena usporedivih stavki?
- Kojem kvartilu ili percentilu pripada pojedina cijena?

## Narudžbe i dostava

- Kolika je vrijednost pojedine narudžbe?
- Koliko traje odobrenje, predaja prijevozniku i dostava?
- Koliki udio narudžbi kasni?
- Kako se trajanje dostave razlikuje prema državi kupca i vremenu?
- Kod kojih se narudžbi plaćeni iznos razlikuje od obračunatog?

## Plaćanja

- Koji se tipovi plaćanja najčešće koriste?
- Koliki su iznosi plaćanja i broj rata?
- Koliko zapisa ima neuobičajene sekvence, nulte iznose ili neispravan broj rata?

## Recenzije

- Kako se ocjene mijenjaju kroz vrijeme?
- Koliki je udio niskih ocjena?
- Koliko traje odgovor na recenziju?
- Postoji li povezanost između dostave i zadovoljstva kupaca?

## Statistička analiza

- Kakva je distribucija cijena unutar kategorije i mjeseca?
- Postoji li povezanost cijene, težine ili volumena proizvoda s troškom dostave?
- Na koliko se opažanja temelji pojedina statistička procjena?

# Odabrani poslovni procesi i tipovi činjenica

| Činjenična tablica | Poslovni proces | Tip činjenice | Granulacija |
|---|---|---|---|
| `fact_sales` | prodaja proizvoda | transaction fact | jedna stavka narudžbe |
| `fact_order_lifecycle` | obrada i dostava narudžbe | accumulating snapshot | jedna narudžba |
| `fact_payments` | evidentiranje plaćanja | transaction fact | jedan zapis plaćanja |
| `fact_reviews` | završna povratna informacija | transaction fact nakon odabira zadnje recenzije | jedna posljednja recenzija narudžbe |
| `fact_category_month_statistics` | mjesečni statistički sažetak prodaje | periodic snapshot / aggregate fact | jedna kategorija u jednom mjesecu |


# Dimenzijske tablice

## `dim_date`

### Granulacija

Jedan redak predstavlja jedan kalendarski datum.

`dim_date` ne sadrži sate, minute ni sekunde. Vremenski dio timestampova modelira se zasebnom dimenzijom `dim_time`.

### Atributi

- `date_key` – surogatni primarni ključ;
- `full_date` – puni kalendarski datum;
- `day_of_month`;
- `day_of_week_iso`;
- `day_name`;
- `day_of_year`;
- `week_of_year`;
- `iso_week_year`;
- `week_start_date`;
- `week_end_date`;
- `month_number`;
- `month_name`;
- `month_start_date`;
- `month_end_date`;
- `quarter_number`;
- `year_number`;
- `is_weekend`;
- `season`;
- `is_unknown`.


Ista dimenzija koristi se u više uloga: datum kupnje, odobrenja, predaje dostavljaču, dostave, očekivane dostave, nastanka recenzije i odgovora na recenziju.

Poseban zapis `Unknown`, primjerice `date_key = 0`, koristi se kada izvorni datum ne postoji. 

Dimenzija se generira za cijelo kalendarsko razdoblje obuhvaćeno podacima, uključujući datume na kojima nije zabilježen nijedan poslovni događaj.

### Role-playing primjena

Ista fizička tablica `dim_date` u činjeničnim se tablicama koristi pod različitim nazivima stranih ključeva, primjerice:

- `purchase_date_key`;
- `approval_date_key`;
- `delivery_date_key`;
- `review_creation_date_key`.

Time nije potrebno izrađivati zasebnu tablicu datuma za svaki događaj.

## `dim_time`

### Granulacija

Jedan redak predstavlja jednu minutu unutar kalendarskog dana.

Dimenzija nije vezana uz određeni datum. Vrijednost `14:35` postoji samo jednom i može se koristiti uz bilo koji `date_key`.

### Broj redaka

Dan sadrži:

`24 × 60 = 1 440`

mogućih minuta.

Uz poseban zapis `Unknown`, dimenzija ima ukupno 1 441 redak. 

### Atributi

- `time_key` – surogatni primarni ključ tipa integer;
- `full_time` – vrijeme tipa `TIME`, primjerice `14:35`;
- `hour_number` – sat od 0 do 23;
- `minute_number` – minuta od 0 do 59;
- `minute_of_day` – redni broj minute unutar dana, od 0 do 1439;
- `part_of_day` – poslovna grupa poput `night`, `morning`, `afternoon` ili `evening`;
- `is_business_hours` – indikator pripada li vrijeme definiranom radnom vremenu;
- `is_unknown` – indikator posebnog nepoznatog vremena.

### Predložena podjela dijela dana

Jedno moguće pravilo je:

- `night`: 00:00–05:59;
- `morning`: 06:00–11:59;
- `afternoon`: 12:00–17:59;
- `evening`: 18:00–23:59.


### Zašto se koristi zasebna dimenzija vremena?

Kada bi se svaki mogući datum i vrijeme spremili u jednu datetime dimenziju, broj redaka rastao bi s brojem dana i vremenskom preciznošću. Razdvajanjem:

- `dim_date` sadrži samo datume;
- `dim_time` sadrži samo trenutke unutar dana;
- činjenični redak kombinira odgovarajući `date_key` i `time_key`.

Tako se može analizirati:

- broj kupnji po satu;
- raspodjela kupnji prema dijelu dana;
- vrijeme odobravanja narudžbi;
- aktivnosti izvan uobičajenog radnog vremena;
- razlike između datuma i vremena pojedinih faza procesa.

## `dim_customer`

### Granulacija

Jedan redak predstavlja jedan izvorni `customer_id`.

`customer_unique_id` ostaje atribut dimenzije i omogućuje povezivanje više narudžbi iste stvarne osobe.

### Atributi

- `customer_key` – surogatni primarni ključ;
- `customer_id` 
- `customer_unique_id`;
- `customer_zip_code_prefix`;
- `customer_city`;
- `customer_state`;
- `customer_city_corrected_flag`;
- `customer_state_corrected_flag`;
- `customer_geolocation_matched_flag`.


## `dim_seller`

### Granulacija

Jedan redak predstavlja jednog prodavača, određenog izvornim `seller_id`.

### Atributi

- `seller_key` – surogatni primarni ključ;
- `seller_id` 
- `seller_zip_code_prefix`;
- `seller_city`;
- `seller_state`;
- `seller_city_corrected_flag`;
- `seller_state_corrected_flag`;
- `seller_geolocation_matched_flag`;
- `seller_city_invalid_format_flag`.


## `dim_product`

### Granulacija

Jedan redak predstavlja jedan proizvod, određen izvornim `product_id`.

### Atributi

- `product_key` – surogatni primarni ključ;
- `product_id` 
- `category_name_portuguese`;
- `category_name_english`;
- `is_unknown_category`;
- `product_weight_g`;
- `product_length_cm`;
- `product_height_cm`;
- `product_width_cm`;
- `product_volume_cm3`;
- `missing_dimensions_flag`;
- `invalid_weight_flag`;


## `dim_category`

### Granulacija

Jedan redak predstavlja jednu standardiziranu kategoriju proizvoda.

### Atributi

- `category_key` – surogatni primarni ključ;
- `category_name_portuguese` – poslovni ključ;
- `category_name_english`;
- `is_unknown_category`.

### Obrazloženje

Ova dimenzija koristi se uz `fact_category_month_statistics`, čija je granulacija kategorija–mjesec.

`dim_product` i dalje sadrži naziv kategorije. `dim_category` nije uvedena radi normaliziranja zvijezde, nego zato što agregirana činjenica nema jedan konkretan proizvod, nego jednu kategoriju.

## `dim_order_status`

### Granulacija

Jedan redak predstavlja jedan standardizirani status narudžbe.

### Atributi

- `order_status_key` – surogatni primarni ključ;
- `order_status` – `created`, `approved`, `invoiced`, `processing`, `shipped`, `delivered`, `canceled` ili `unavailable`;
- `status_group` – npr. aktivan, uspješno završen ili neuspješno završen;
- `status_phase_order` – poslovni redoslijed statusa kada je primjenjiv;
- `is_final_status`;
- `is_successful_status`.


## `dim_payment_type`

### Granulacija

Jedan redak predstavlja jedan standardizirani tip plaćanja.

### Atributi

- `payment_type_key` – surogatni primarni ključ;
- `payment_type` – `credit_card`, `debit_card`, `boleto`, `voucher` ili `not_defined`;
- `is_card_payment`;
- `supports_installments`.


# Činjenična tablica `fact_sales`

## Poslovni proces

Proces je prodaja proizvoda putem narudžbe. Događaj nastaje kada je proizvod evidentiran kao zasebna stavka narudžbe kod određenog prodavača.

## Tip i granulacija

Ovo je transakcijska činjenična tablica.

**Jedan redak predstavlja jednu izvornu stavku proizvoda u jednoj narudžbi.**

Kontrola granulacije:

`UNIQUE (order_id, order_item_id)`.

Ista narudžba može sadržavati više jednakih proizvoda. Svaki takav proizvod ima različit `order_item_id` i predstavlja zaseban redak.

## Ključevi

- `sales_key` – surogatni primarni ključ;
- `purchase_date_key` – FK prema `dim_date`;
- `purchase_time_key` – FK prema `dim_time`;
- `shipping_limit_date_key` – FK prema `dim_date`;
- `shipping_limit_time_key` – FK prema `dim_time`;
- `customer_key` – FK prema `dim_customer`;
- `seller_key` – FK prema `dim_seller`;
- `product_key` – FK prema `dim_product`;
- `order_status_key` – FK prema `dim_order_status`;
- `order_id` 
- `order_item_id` 

`shipping_limit_date_key` i `shipping_limit_time_key` nastaju rastavljanjem izvornog `shipping_limit_date` timestampa. Omogućuju analizu roka predaje stavke i prema datumu i prema dobu dana.

`purchase_date_key` i `purchase_time_key` izvedeni su iz `order_purchase_timestamp` pripadne narudžbe.

## Osnovne mjere

- `item_count` – uvijek 1;
- `product_value` – izvorni `price`;
- `freight_value`;
- `item_total_value` – `product_value + freight_value`.


## Indikatori kvalitete i poslovnih posebnosti

- `free_freight_flag`;
- `is_outlier_price`;
- `freight_outlier_flag`;
- `freight_greater_than_price_flag`.

### Detekcija odstupajućih cijena IQR metodom

Za svaku prodajnu stavku određuje se pripada li njezina cijena odstupajućim vrijednostima primjenom IQR metode.

Referentnu skupinu čine sve prodajne stavke iste kategorije proizvoda u istom kalendarskom mjesecu.

Za svaku referentnu skupinu računaju se prvi i treći kvartil:

`IQR = Q3 - Q1`

Donja i gornja granica određuju se kao:

- `lower_bound = Q1 - 1.5 × IQR`;
- `upper_bound = Q3 + 1.5 × IQR`.

Vrijedi:

`is_outlier_price = TRUE`

ako je:

`product_value < lower_bound`

ili

`product_value > upper_bound`.

Ovaj indikator ne znači da je podatak pogrešan. Odstupajuće vrijednosti zadržavaju se u skladištu te indikator omogućuje njihovo izdvajanje i zasebnu analizu.

## Statistički izvedeni atributi cijene

Referentna skupina:

**sve prodajne stavke iste kategorije proizvoda u istom kalendarskom mjesecu**.

- `category_month_price_mean`;
- `category_month_price_median`;
- `category_month_price_stddev`;
- `price_deviation_from_mean`;
- `price_quartile`;
- `price_percentile`;
- `price_reference_group_size`;
- `price_statistics_valid_flag`.

Odstupanje:

`product_value - category_month_price_mean`.


# Činjenična tablica `fact_order_lifecycle`

## Poslovni proces

Proces je obrada i dostava narudžbe od kupnje do završnog statusa.

## Tip i granulacija

Ovo je accumulating snapshot.

**Jedan redak predstavlja jednu narudžbu tijekom cijelog životnog ciklusa.**

Kontrola granulacije:

`UNIQUE (order_id)`.

Proizvod i prodavač nisu izravne dimenzije ove tablice jer jedna narudžba može sadržavati više proizvoda i više prodavača.

## Ključevi

- `order_lifecycle_key` – surogatni primarni ključ;
- `customer_key` – FK prema `dim_customer`;
- `order_status_key` – FK prema `dim_order_status`;
- `purchase_date_key` – FK prema `dim_date`;
- `purchase_time_key` – FK prema `dim_time`;
- `approval_date_key` – FK prema `dim_date`;
- `approval_time_key` – FK prema `dim_time`;
- `carrier_date_key` – FK prema `dim_date`;
- `carrier_time_key` – FK prema `dim_time`;
- `delivery_date_key` – FK prema `dim_date`;
- `delivery_time_key` – FK prema `dim_time`;
- `estimated_delivery_date_key` – FK prema `dim_date`;
- `estimated_delivery_time_key` – FK prema `dim_time`;
- `order_id` 

Svaki izvorni timestamp rastavlja se na odgovarajući datumski i vremenski ključ. Na primjer:

- `order_purchase_timestamp` → `purchase_date_key` + `purchase_time_key`;
- `order_approved_at` → `approval_date_key` + `approval_time_key`;
- `order_delivered_customer_date` → `delivery_date_key` + `delivery_time_key`.

Ako timestamp nedostaje, oba ključa pokazuju na `Unknown` zapise.


## Financijske i brojčane mjere

- `order_count` – uvijek 1;
- `item_count`;
- `product_value`;
- `freight_value`;
- `order_total_value`;
- `payment_total_value`;
- `payment_difference_value`;
- `payment_mismatch_flag`.

Formule:

- `order_total_value = product_value + freight_value`;
- `payment_difference_value = payment_total_value - order_total_value`;
- `payment_mismatch_flag = TRUE` kada je apsolutna razlika najmanje 0,01.

U čišćenju je identificirano 576 takvih narudžbi. Indikator se sprema ovdje, a ne u `fact_payments`, jer je odstupanje svojstvo cijele narudžbe.

## Vremenske mjere

- `approval_time_hours`;
- `carrier_wait_days`;
- `shipping_time_days`;
- `delivery_time_days`;
- `delivery_delay_days`;
- `delivered_within_deadline_flag`.

## Indikatori vremenskih anomalija

- `approval_before_purchase_flag`;
- `carrier_before_approval_flag`;
- `customer_before_carrier_flag`;
- `delivery_before_purchase_flag`;
- `has_timeline_anomaly_flag`.

U podacima postoji 1 382 narudžbe s barem jednom takvom anomalijom. Zapisi se zadržavaju te se mogu uključiti ili isključiti iz analize.

## Statistički izvedeni atributi

### Vrijednost narudžbe

Referentnu skupinu čine:

**sve narudžbe kupljene u istom kalendarskom mjesecu**.

Za svaku narudžbu spremaju se:

- `month_order_value_mean`;
- `month_order_value_median`;
- `month_order_value_stddev`;
- `order_value_deviation_from_mean`;
- `order_value_quartile`;
- `order_value_percentile`;
- `order_value_reference_group_size`;
- `order_value_statistics_valid_flag`;
- `is_outlier_order_value`.

`is_outlier_order_value` označava je li ukupna vrijednost narudžbe
(`order_total_value`) statistički odstupajuća unutar pripadajućeg
kalendarskog mjeseca prema IQR metodi.

Za referentnu skupinu računaju se prvi i treći kvartil:

`IQR = Q3 - Q1`

te granice:

- `lower_bound = Q1 - 1.5 × IQR`;
- `upper_bound = Q3 + 1.5 × IQR`.

Vrijedi `is_outlier_order_value = TRUE` ako je `order_total_value`
manji od donje ili veći od gornje granice.

### Trajanje dostave

- `is_outlier_delivery_time` – označava je li ukupno vrijeme dostave
  (`delivery_time_days`) statistički odstupajuće prema IQR metodi.

Referentnu skupinu čine sve narudžbe za koje je poznato i valjano
`delivery_time_days`.

Za referentnu skupinu računaju se:

`IQR = Q3 - Q1`

te granice:

- `lower_bound = Q1 - 1.5 × IQR`;
- `upper_bound = Q3 + 1.5 × IQR`.

Vrijedi `is_outlier_delivery_time = TRUE` ako je `delivery_time_days`
manji od donje ili veći od gornje granice.

Narudžbe za koje nije moguće izračunati trajanje dostave ne sudjeluju
u određivanju IQR granica, a njihov `is_outlier_delivery_time` ostaje
`NULL`.

Zapisi s prethodno identificiranim anomalijama vremenskog slijeda
zadržavaju se u skladištu i označeni su zasebnim indikatorima kvalitete.

Označavanje vrijednosti kao statističkog outliera ne znači da je zapis
pogrešan. Outlieri se ne uklanjaju iz skladišta, nego se indikatorima
omogućuje njihovo jednostavno izdvajanje i daljnja analiza.


# Činjenična tablica `fact_payments`

## Poslovni proces

Proces je evidentiranje načina i iznosa kojima je kupac podmirio narudžbu.

## Tip i granulacija

Ovo je transakcijska činjenična tablica.

**Jedan redak predstavlja jedan izvorni zapis plaćanja unutar jedne narudžbe.**

Kontrola granulacije:

`UNIQUE (order_id, payment_sequential)`.

## Ključevi

- `payment_key` – surogatni primarni ključ;
- `purchase_date_key` – FK prema `dim_date`;
- `purchase_time_key` – FK prema `dim_time`;
- `customer_key` – FK prema `dim_customer`;
- `payment_type_key` – FK prema `dim_payment_type`;
- `order_id` 
- `payment_sequential`.

`purchase_date_key` i `purchase_time_key` predstavljaju datum i vrijeme kupnje pripadne narudžbe jer izvorna tablica plaćanja nema vlastiti timestamp. Ti ključevi ne predstavljaju stvarno vrijeme izvršenja plaćanja, nego vremenski kontekst narudžbe.

## Mjere

- `payment_value`;
- `payment_installments`.


## Indikatori kvalitete

- `payment_value_zero_flag`;
- `installments_nonpositive_flag`;
- `payment_sequence_anomaly_flag`;
- `payment_value_outlier_flag`.

`payment_amount_mismatch` se ne prenosi u ovu činjenicu jer bi se ponovio na svim zapisima plaćanja iste narudžbe. Pohranjuje se samo u `fact_order_lifecycle`.

# Činjenična tablica `fact_reviews`

## Poslovni proces

Proces je evidentiranje posljednje poznate povratne informacije kupca o narudžbi.

## Tip i granulacija

**Jedan redak predstavlja posljednju dostupnu recenziju jedne narudžbe.**

Kontrola granulacije:

`UNIQUE (order_id)`.

Tijekom čišćenja iz 99 224 izvorna retka dobiveno je 98 673 posljednjih recenzija, bez dupliciranih `order_id`.

## Ključevi

- `review_key` – surogatni primarni ključ;
- `customer_key` – FK prema `dim_customer`;
- `purchase_date_key` – FK prema `dim_date`;
- `purchase_time_key` – FK prema `dim_time`;
- `review_creation_date_key` – FK prema `dim_date`;
- `review_creation_time_key` – FK prema `dim_time`;
- `review_answer_date_key` – FK prema `dim_date`;
- `review_answer_time_key` – FK prema `dim_time`;
- `order_status_key` – FK prema `dim_order_status`;
- `order_id` 
- `review_id` 

`review_creation_date_key` i `review_creation_time_key` dobivaju se iz `review_creation_date`, dok se ključevi odgovora dobivaju iz `review_answer_timestamp`.

## Mjere i indikatori

- `review_score`;
- `review_response_time_hours`;
- `has_review_title_flag`;
- `has_review_message_flag`;
- `low_score_flag`;
- `multiple_reviews_source_flag`.

`multiple_reviews_source_flag` označava da je narudžba prije odabira posljednje recenzije imala više zapisa. Takvih je narudžbi 547.

Ne dodaju se `product_key`, `seller_key` ni `category_key`. Recenzija pripada cijeloj narudžbi, koja može sadržavati više proizvoda i prodavača. 

# Činjenična tablica `fact_category_month_statistics`

## Poslovni proces

Proces je mjesečno statističko praćenje prodaje, cijena i troškova dostave
po kategoriji proizvoda.

Tablica omogućuje unaprijed izračunate statističke pokazatelje na razini
kategorije proizvoda i kalendarskog mjeseca. Time se mjere lokacije,
raspršenosti i korelacije uključuju izravno u proces izgradnje skladišta
podataka, umjesto da se svaki put iznova računaju prilikom analize.

Primjeri poslovnih pitanja na koja tablica omogućuje odgovor su:

- Kako se prosječna i medijalna cijena proizvoda pojedine kategorije
  mijenjaju kroz mjesece?
- Koje kategorije imaju najveću varijabilnost cijena i troškova dostave?
- Postoji li unutar pojedine kategorije povezanost između cijene proizvoda
  i troška dostave te mijenja li se ta povezanost kroz vrijeme?
- Postoji li povezanost između težine proizvoda i troška dostave?
- Postoji li povezanost između volumena proizvoda i troška dostave?
- Na koliko se opažanja temelji pojedina procjena korelacije?

## Tip i granulacija

Ovo je periodic snapshot / aggregate fact table.

**Jedan redak predstavlja jednu kategoriju proizvoda u jednom kalendarskom mjesecu.**

Kontrola granulacije:

`UNIQUE (month_date_key, category_key)`.

## Ključevi

- `category_month_statistics_key` – surogatni primarni ključ;
- `month_date_key` – FK prema prvom danu mjeseca u `dim_date`;
- `category_key` – FK prema `dim_category`.

Ova tablica nema `time_key` jer je njezina vremenska granulacija cijeli
kalendarski mjesec, a ne trenutak unutar dana. `month_date_key` pokazuje
na prvi dan pripadnog mjeseca.

## Aditivne mjere

- `sales_item_count`;
- `product_value_sum`;
- `freight_value_sum`;
- `item_total_value_sum`.

Navedene mjere mogu se zbrajati kroz kategorije i vremenska razdoblja.

## Mjere lokacije i raspršenosti

Za cijenu proizvoda spremaju se:

- `price_mean`;
- `price_median`;
- `price_stddev`;
- `price_min`;
- `price_max`;
- `price_q1`;
- `price_q3`;
- `price_iqr`.

Za trošak dostave spremaju se:

- `freight_mean`;
- `freight_median`;
- `freight_stddev`;
- `freight_q1`;
- `freight_q3`;
- `freight_iqr`.

Vrijedi:

`price_iqr = price_q3 - price_q1`

i

`freight_iqr = freight_q3 - freight_q1`.

Ove statističke mjere nisu aditivne. Primjerice, prosjek cijene za više
mjeseci ne dobiva se zbrajanjem ili jednostavnim prosjekom spremljenih
mjesečnih prosjeka. Za izračun na višoj razini agregacije potrebno je
koristiti odgovarajuće osnovne mjere i broj opažanja.

## Korelacijske mjere

U tablici se spremaju korelacije između obilježja koja postoje na
granulaciji pojedinačne prodajne stavke.

Za svaku kategoriju proizvoda i kalendarski mjesec računaju se:

- `corr_price_freight` – Pearsonov koeficijent korelacije između cijene
  proizvoda i troška dostave;
- `corr_price_freight_n` – broj valjanih parova opažanja korištenih za
  izračun korelacije;
- `corr_price_freight_valid_flag` – indikator je li korelaciju bilo moguće
  valjano izračunati;

- `corr_weight_freight` – Pearsonov koeficijent korelacije između težine
  proizvoda i troška dostave;
- `corr_weight_freight_n` – broj valjanih parova opažanja korištenih za
  izračun korelacije;
- `corr_weight_freight_valid_flag` – indikator je li korelaciju bilo moguće
  valjano izračunati;

- `corr_volume_freight` – Pearsonov koeficijent korelacije između volumena
  proizvoda i troška dostave;
- `corr_volume_freight_n` – broj valjanih parova opažanja korištenih za
  izračun korelacije;
- `corr_volume_freight_valid_flag` – indikator je li korelaciju bilo moguće
  valjano izračunati.

Korelacije se računaju nad svim valjanim prodajnim stavkama koje pripadaju
istoj kategoriji proizvoda i istom kalendarskom mjesecu.

Računaju se:

- `corr(product_value, freight_value)`;
- `corr(product_weight_g, freight_value)`;
- `corr(product_volume_cm3, freight_value)`.

Za izračun se koristi Pearsonov koeficijent korelacije.

Broj valjanih parova sprema se zasebno za svaku korelaciju jer se broj
opažanja može razlikovati. Primjerice, prodajna stavka može imati poznatu
cijenu i trošak dostave, dok za pripadajući proizvod može nedostajati
valjana vrijednost težine ili volumena.

Korelacijski koeficijenti nisu aditivne mjere te se ne smiju zbrajati
niti jednostavno prosječiti između različitih kategorija ili mjeseci.